# 🤖 MiniGPT — Treino Completo
JAX/Flax NNX | TinyStories | Orbax Checkpoints | Safetensors Export

In [ ]:
# ── Célula 1 — Instalação ─────────────────────────────────────────────────────
!pip install -q flax==0.10.2 optax==0.2.3 orbax-checkpoint==0.6.4 tiktoken==0.8.0 safetensors==0.4.5 gradio datasets


In [ ]:
# ── Célula 2 — Imports ────────────────────────────────────────────────────────
import os
import time
import jax
import jax.numpy as jnp
from flax import nnx
import optax
import orbax.checkpoint as ocp
import tiktoken
import numpy as np
from datasets import load_dataset
from pathlib import Path


In [ ]:
# ── Célula 3 — Hiperparâmetros ────────────────────────────────────────────────
tokenizer = tiktoken.get_encoding('gpt2')

VOCAB_SIZE             = tokenizer.n_vocab        # 50 257
NUM_TRANSFORMER_BLOCKS = 6
MAXLEN                 = 128
EMBED_DIM              = 384
NUM_HEADS              = 6
FEED_FORWARD_DIM       = int(2/3 * 4 * EMBED_DIM)  # 1024

BATCH_SIZE             = 32
NUM_EPOCHS             = 5
LEARNING_RATE          = 3e-4

MAX_STORIES            = 150_000

assert EMBED_DIM % NUM_HEADS == 0
print('✅ Hiperparâmetros definidos')
print(f'   EMBED_DIM={EMBED_DIM} | NUM_HEADS={NUM_HEADS} | BLOCKS={NUM_TRANSFORMER_BLOCKS}')
print(f'   BATCH_SIZE={BATCH_SIZE} | EPOCHS={NUM_EPOCHS} | MAX_STORIES={MAX_STORIES:,}')


In [ ]:
# ── Célula 4 — Arquitectura do Modelo ────────────────────────────────────────
class Embedding(nnx.Module):
    def __init__(self, vocab_size, d_model, max_len, rngs):
        self.token_emb = nnx.Embed(vocab_size, d_model, rngs=rngs)
        self.pos_emb   = nnx.Embed(max_len, d_model, rngs=rngs)

    def __call__(self, x):
        b, t = x.shape
        pos = jnp.arange(t)[None, :]
        return self.token_emb(x) + self.pos_emb(pos)


class TransformerBlock(nnx.Module):
    def __init__(self, d_model, n_heads, rngs):
        self.attention = nnx.MultiHeadAttention(
            num_heads=n_heads,
            in_features=d_model,
            decode=False,
            rngs=rngs
        )
        self.ffn = nnx.Sequential(
            nnx.Linear(d_model, 4 * d_model, rngs=rngs),
            nnx.gelu,
            nnx.Linear(4 * d_model, d_model, rngs=rngs)
        )
        self.ln1 = nnx.LayerNorm(d_model, rngs=rngs)
        self.ln2 = nnx.LayerNorm(d_model, rngs=rngs)

    def __call__(self, x):
        x = x + self.attention(self.ln1(x))
        x = x + self.ffn(self.ln2(x))
        return x


class MiniGPT(nnx.Module):
    def __init__(self, vocab_size, d_model, n_heads, n_layers, max_len, rngs):
        self.embedding = Embedding(vocab_size, d_model, max_len, rngs)
        self.transformer_blocks = [
            TransformerBlock(d_model, n_heads, rngs) for _ in range(n_layers)
        ]
        self.output_layer = nnx.Linear(d_model, vocab_size, use_bias=False, rngs=rngs)

    def __call__(self, x):
        x = self.embedding(x)
        for block in self.transformer_blocks:
            x = block(x)
        return self.output_layer(x)


rngs  = nnx.Rngs(0)
model = MiniGPT(VOCAB_SIZE, EMBED_DIM, NUM_HEADS, NUM_TRANSFORMER_BLOCKS, MAXLEN, rngs)
print('✅ Modelo criado')


In [ ]:
# ── Célula 5 — Dataset e Batches ─────────────────────────────────────────────
print('📦 A carregar dataset TinyStories...')
ds = load_dataset('roneneldan/TinyStories', split='train')
ds = ds.select(range(MAX_STORIES))

def tokenize(example):
    return tokenizer.encode(example['text'])[:MAXLEN + 1]

print('🔄 A tokenizar...')
all_tokens = []
for i, example in enumerate(ds):
    tokens = tokenize(example)
    if len(tokens) >= 2:
        all_tokens.append(tokens)
    if (i + 1) % 10000 == 0:
        print(f'   {i+1:,}/{MAX_STORIES:,} histórias processadas')

def pad_or_truncate(seq, length):
    if len(seq) >= length + 1:
        return seq[:length], seq[1:length+1]
    pad = [0] * (length + 1 - len(seq))
    seq = seq + pad
    return seq[:length], seq[1:length+1]

X_list, Y_list = [], []
for tokens in all_tokens:
    x, y = pad_or_truncate(tokens, MAXLEN)
    X_list.append(x)
    Y_list.append(y)

X = jnp.array(X_list)
Y = jnp.array(Y_list)

n_batches = len(X) // BATCH_SIZE
X = X[:n_batches * BATCH_SIZE].reshape(n_batches, BATCH_SIZE, MAXLEN)
Y = Y[:n_batches * BATCH_SIZE].reshape(n_batches, BATCH_SIZE, MAXLEN)
batches = list(zip(X, Y))

print(f'✅ {len(batches):,} batches criados ({len(X_list):,} sequências)')


In [ ]:
# ── Célula 6 — Optimizer e Train Step ────────────────────────────────────────
optimizer = nnx.Optimizer(model, optax.adam(LEARNING_RATE), wrt=nnx.Param)

@nnx.jit
def train_step(model, optimizer, x, y):
    def loss_fn(model):
        logits = model(x)
        loss = optax.softmax_cross_entropy_with_integer_labels(
            logits.reshape(-1, VOCAB_SIZE),
            y.reshape(-1)
        ).mean()
        return loss

    loss, grads = nnx.value_and_grad(loss_fn)(model)
    optimizer.update(grads)
    return loss

print('✅ Optimizer e train_step prontos')


In [ ]:
# ── Célula 7 — Google Drive + Retoma ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')
CHECKPOINT_DIR = '/content/drive/MyDrive/minigpt_treino'
os.makedirs(CHECKPOINT_DIR, exist_ok=True)

RESUME_FROM_EPOCH = 0   # muda para o número do último epoch guardado

if RESUME_FROM_EPOCH > 0:
    resume_path = os.path.join(CHECKPOINT_DIR, f'epoch_{RESUME_FROM_EPOCH}')
    if os.path.exists(resume_path):
        print(f'🔄 A retomar do epoch {RESUME_FROM_EPOCH}...')
        _, state = nnx.split(model)
        state = ocp.PyTreeCheckpointer().restore(resume_path, item=state)
        nnx.update(model, state)
        print(f'✅ Pesos carregados de: {resume_path}')
    else:
        print(f'❌ Checkpoint não encontrado. A iniciar do zero.')
        RESUME_FROM_EPOCH = 0
else:
    print('🆕 A iniciar treino do zero.')

print(f'   Checkpoint dir: {CHECKPOINT_DIR}')


In [ ]:
# ── Célula 8 — Loop de Treino ─────────────────────────────────────────────────
print(f'🚀 Treino iniciado!')
print(f'   Epochs:         {NUM_EPOCHS}')
print(f'   Batches/epoch:  {len(batches):,}')
print(f'   Total de steps: {NUM_EPOCHS * len(batches):,}')
print()

history = []

for epoch in range(RESUME_FROM_EPOCH, NUM_EPOCHS):
    epoch_loss = 0.0
    t_start    = time.time()

    print(f'── Epoch {epoch+1}/{NUM_EPOCHS} ──────────────────────────────────────')

    for step, (x_batch, y_batch) in enumerate(batches):
        loss        = train_step(model, optimizer, x_batch, y_batch)
        loss_val    = float(loss)
        epoch_loss += loss_val

        if (step + 1) % 200 == 0:
            avg     = epoch_loss / (step + 1)
            elapsed = time.time() - t_start
            steps_done      = step + 1
            steps_remaining = (len(batches) - steps_done) + (NUM_EPOCHS - epoch - 1) * len(batches)
            eta_min = elapsed / steps_done * steps_remaining / 60
            print(f'   Step {step+1:>5}/{len(batches)} | Loss: {avg:.4f} | ETA: {eta_min:.0f} min')

    avg_loss      = epoch_loss / len(batches)
    elapsed_total = (time.time() - t_start) / 60
    history.append(avg_loss)
    print(f'   ✅ Epoch {epoch+1} — Loss: {avg_loss:.4f} | Tempo: {elapsed_total:.1f} min')

    save_path = os.path.join(CHECKPOINT_DIR, f'epoch_{epoch+1}')
    os.makedirs(save_path, exist_ok=True)
    _, state = nnx.split(model)
    ocp.PyTreeCheckpointer().save(save_path, state, force=True)
    print(f'   💾 Checkpoint guardado: {save_path}')
    print()

print(f'🎉 Treino concluído!')
print(f'   Loss inicial: {history[0]:.4f}')
print(f'   Loss final:   {history[-1]:.4f}')


In [ ]:
# ── Célula 9 — Exportar para Safetensors ─────────────────────────────────────
from safetensors.numpy import save_file

print('📦 A exportar pesos para Safetensors...')
_, state = nnx.split(model)
flat = state.flat_state()

weights = {}
for key_tuple, var in flat.items():
    key_str = '.'.join(str(k) for k in key_tuple)
    weights[key_str] = np.array(var.value)

save_file(weights, 'minigpt_weights.safetensors')
print(f'✅ Exportado: minigpt_weights.safetensors ({len(weights)} tensores)')
print('   Faz o download e faz upload para o teu Hugging Face Space.')


In [ ]:
# ── Célula 10 — Teste de Geração ──────────────────────────────────────────────
def generate_text(prompt, max_new_tokens=50, temperature=0.7, top_k=40):
    input_ids = jnp.array([tokenizer.encode(prompt)])

    for _ in range(int(max_new_tokens)):
        curr_input = input_ids[:, -MAXLEN:]
        logits = model(curr_input)[:, -1, :]
        logits = logits / float(temperature)

        top_values, _ = jax.lax.top_k(logits, int(top_k))
        logits = jnp.where(logits < top_values[:, -1:], -jnp.inf, logits)

        key = jax.random.PRNGKey(int(time.time_ns() % (2**31)))
        next_token = jax.random.categorical(key, logits)[:, None]
        input_ids = jnp.concatenate([input_ids, next_token], axis=1)

        if int(next_token[0, 0]) == tokenizer.eot_token:
            break

    return tokenizer.decode(input_ids[0].tolist())

# Teste
print(generate_text('Once upon a time'))
